# Ricci Finance v13 — Regime Engine Benchmark
## hmmlearn vs pomegranate vs duration-aware HSMM

This notebook compares three regime engines on the same standardized Ricci-capital feature matrix.

1. **hmmlearn GaussianHMM** — conventional baseline.
2. **pomegranate DenseHMM** — PyTorch-based HMM with posterior probabilities and optional GPU execution.
3. **Duration-aware HSMM** — explicit-duration Gaussian semi-Markov decoder that penalizes unrealistic one-frame switching.

Validation has two layers:
- **Synthetic labeled regimes:** accuracy, macro-F1, adjusted Rand index, duration recovery.
- **v12 Ricci features:** walk-forward stability, switch rate, run length, posterior entropy, runtime, and regime separation.

> Hidden-state numbers are arbitrary. Predicted labels are aligned with the Hungarian algorithm before labeled evaluation.

## Method comparison

| Method | Advantage | Limitation | v13 role |
|---|---|---|---|
| hmmlearn | Simple and compatible with v12 | CPU-oriented; limited-maintenance | Baseline |
| pomegranate | PyTorch/GPU, flexible distributions and masks | Different API; current v1 lacks Viterbi | Extensible engine |
| HSMM | Explicit regime duration | More computation; posterior probability not yet implemented here | Duration-aware engine |

In [ ]:
# Optional in a fresh environment
# %pip install -r requirements-v13.txt

In [ ]:
from __future__ import annotations
import math, time, warnings
from dataclasses import dataclass
from typing import Any
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.stats import multivariate_normal
from sklearn.metrics import accuracy_score, adjusted_rand_score, f1_score
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)
print("NumPy",np.__version__,"Pandas",pd.__version__)

In [ ]:
@dataclass
class RegimeResult:
    method:str
    states:np.ndarray
    probabilities:np.ndarray|None
    runtime_seconds:float
    model:Any
    available:bool=True
    message:str=""

def run_lengths(states):
    states=np.asarray(states,dtype=int)
    if len(states)==0:return np.array([],dtype=int)
    cuts=np.flatnonzero(np.r_[True,states[1:]!=states[:-1],True])
    return np.diff(cuts)

def switch_rate(states):
    states=np.asarray(states)
    return float(np.mean(states[1:]!=states[:-1])) if len(states)>1 else 0.0

def posterior_entropy(p):
    if p is None:return float("nan")
    p=np.clip(np.asarray(p,float),1e-12,1)
    return float(np.mean(-np.sum(p*np.log(p),axis=1)))

def align_states(reference,predicted):
    reference=np.asarray(reference,int); predicted=np.asarray(predicted,int)
    R=np.unique(reference); P=np.unique(predicted)
    cost=np.zeros((len(P),len(R)))
    for i,p in enumerate(P):
        for j,r in enumerate(R): cost[i,j]=-np.sum((predicted==p)&(reference==r))
    row,col=linear_sum_assignment(cost)
    mapping={int(P[i]):int(R[j]) for i,j in zip(row,col)}
    return np.array([mapping.get(int(s),int(s)) for s in predicted]),mapping

In [ ]:

def compare_viterbi_and_posterior(
    X: np.ndarray,
    reference_states: np.ndarray | None = None,
    n_states: int = 3,
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Compare global Viterbi decoding with per-frame posterior argmax.

    Uses hmmlearn because it exposes both:
      - predict(X): Viterbi path
      - predict_proba(X): posterior probabilities
    """
    from hmmlearn.hmm import GaussianHMM

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        min_covar=1e-4,
        n_iter=500,
        random_state=RANDOM_STATE,
    )
    model.fit(X)

    viterbi_states = model.predict(X)
    probabilities = model.predict_proba(X)
    posterior_states = probabilities.argmax(axis=1)

    sequences = {
        "Viterbi decoding": viterbi_states,
        "Posterior argmax": posterior_states,
    }

    rows = []
    for name, states in sequences.items():
        lengths = run_lengths(states)
        row = {
            "decoder": name,
            "switch_rate": switch_rate(states),
            "median_run_length": float(np.median(lengths)),
            "mean_run_length": float(np.mean(lengths)),
            "short_run_ratio_le_2": float(np.mean(lengths <= 2)),
            "posterior_entropy": posterior_entropy(probabilities),
        }

        if reference_states is not None:
            aligned, mapping = align_states(reference_states, states)
            row.update({
                "accuracy": accuracy_score(reference_states, aligned),
                "macro_f1": f1_score(reference_states, aligned, average="macro"),
                "adjusted_rand": adjusted_rand_score(reference_states, states),
                "state_mapping": mapping,
            })

        rows.append(row)

    agreement = float(np.mean(viterbi_states == posterior_states))
    comparison = pd.DataFrame(rows)
    comparison["decoder_agreement"] = agreement

    return comparison, sequences


## 1. Synthetic regime data with known truth

In [ ]:
def simulate_regime_features(n_samples=420,random_state=42):
    rng=np.random.default_rng(random_state)
    duration_means=np.array([24,13,38])
    means=np.array([[-.24,.56,.18,.42,.72,.60],[-.05,.34,.29,.56,.48,.42],[.16,.15,.48,.79,.25,.29]])
    stds=np.array([[.09,.10,.07,.10,.12,.10],[.08,.10,.08,.10,.11,.11],[.07,.07,.08,.08,.08,.09]])
    states=[]; current=2
    while len(states)<n_samples:
        d=max(4,int(rng.poisson(duration_means[current]))); states.extend([current]*d)
        current=int(rng.choice([s for s in range(3) if s!=current]))
    states=np.asarray(states[:n_samples],int)
    X=np.vstack([rng.normal(means[s],stds[s]) for s in states])
    X[1:]=.78*X[1:]+.22*X[:-1]
    idx=rng.choice(n_samples,10,replace=False); X[idx]+=rng.normal(0,.22,size=(10,X.shape[1]))
    cols=['avg_ricci','negative_edge_ratio','density','largest_component_ratio','max_node_capital_share','edge_instability']
    dates=pd.bdate_range('2024-01-02',periods=n_samples)
    return pd.DataFrame(X,index=dates,columns=cols),states
synthetic_df,true_states=simulate_regime_features()
synthetic_df.head(),np.bincount(true_states)

In [ ]:
fig,ax=plt.subplots(2,1,figsize=(14,7),sharex=True)
ax[0].plot(synthetic_df.index,synthetic_df.avg_ricci,label='avg_ricci');ax[0].plot(synthetic_df.index,synthetic_df.density,label='density');ax[0].legend();ax[0].set_title('Synthetic observations')
ax[1].step(synthetic_df.index,true_states,where='post');ax[1].set_yticks([0,1,2],['stress','transition','coherent']);ax[1].set_title('Known regime')
plt.tight_layout();plt.show()

## 2. Regime engines

In [ ]:
def fit_hmmlearn(X,n_states=3):
    t=time.perf_counter()
    try:
        from hmmlearn.hmm import GaussianHMM
        m=GaussianHMM(n_components=n_states,covariance_type='diag',min_covar=1e-4,n_iter=500,random_state=RANDOM_STATE)
        m.fit(X); s=m.predict(X); p=m.predict_proba(X)
        return RegimeResult('hmmlearn GaussianHMM',s,p,time.perf_counter()-t,m)
    except Exception as e:return RegimeResult('hmmlearn GaussianHMM',np.full(len(X),-1),None,time.perf_counter()-t,None,False,str(e))

def fit_pomegranate(X,n_states=3):
    t=time.perf_counter()
    try:
        import torch
        from pomegranate.hmm import DenseHMM
        from pomegranate.distributions import Normal
        T=torch.tensor(X[None,:,:],dtype=torch.float32)
        m=DenseHMM([Normal() for _ in range(n_states)],max_iter=250,tol=1e-4,verbose=False)
        m.fit(T); p=m.predict_proba(T)[0].detach().cpu().numpy(); s=p.argmax(axis=1)
        return RegimeResult('pomegranate DenseHMM',s,p,time.perf_counter()-t,m)
    except Exception as e:return RegimeResult('pomegranate DenseHMM',np.full(len(X),-1),None,time.perf_counter()-t,None,False,str(e))

In [ ]:
class ExplicitDurationGaussianHSMM:
    def __init__(self,n_states=3,min_duration=3,max_duration=60,random_state=42):
        self.n_states=n_states;self.min_duration=min_duration;self.max_duration=max_duration;self.random_state=random_state
    def fit(self,X):
        from hmmlearn.hmm import GaussianHMM
        b=GaussianHMM(n_components=self.n_states,covariance_type='diag',min_covar=1e-4,n_iter=500,random_state=self.random_state).fit(X)
        z=b.predict(X);self.means_=np.asarray(b.means_);self.covars_=np.asarray(b.covars_);self.startprob_=np.clip(b.startprob_,1e-12,1);self.transmat_=np.clip(b.transmat_,1e-12,1)
        D={s:[] for s in range(self.n_states)};i=0
        while i<len(z):
            j=i+1
            while j<len(z) and z[j]==z[i]:j+=1
            D[int(z[i])].append(j-i);i=j
        self.duration_lambda_=np.array([max(self.min_duration,np.mean(D[s]) if D[s] else 8.) for s in range(self.n_states)])
        return self
    def emission(self,X):
        out=np.empty((len(X),self.n_states))
        for s in range(self.n_states):
            cov=np.diag(np.maximum(self.covars_[s],1e-5));out[:,s]=multivariate_normal.logpdf(X,mean=self.means_[s],cov=cov,allow_singular=True)
        return out
    def dlog(self,s,d):
        if d<self.min_duration or d>self.max_duration:return -np.inf
        lam=max(self.duration_lambda_[s]-self.min_duration,1e-6);k=d-self.min_duration
        return k*math.log(lam)-lam-math.lgamma(k+1)
    def predict(self,X):
        X=np.asarray(X,float);T=len(X);E=self.emission(X);C=np.vstack([np.zeros(self.n_states),np.cumsum(E,axis=0)])
        score=np.full((T+1,self.n_states),-np.inf);bs=np.full((T+1,self.n_states),-1,int);bd=np.zeros((T+1,self.n_states),int)
        ls=np.log(self.startprob_);lt=np.log(self.transmat_)
        for end in range(1,T+1):
            for s in range(self.n_states):
                for d in range(self.min_duration,min(self.max_duration,end)+1):
                    begin=end-d;seg=C[end,s]-C[begin,s];dl=self.dlog(s,d)
                    if begin==0:cand=ls[s]+seg+dl;prev=-1
                    else:
                        q=score[begin]+lt[:,s];q[s]=-np.inf;prev=int(np.argmax(q));cand=q[prev]+seg+dl
                    if cand>score[end,s]:score[end,s]=cand;bs[end,s]=prev;bd[end,s]=d
        z=np.empty(T,int);end=T;s=int(np.argmax(score[T]))
        while end>0:
            d=int(bd[end,s]) or min(end,self.min_duration);begin=max(0,end-d);z[begin:end]=s;s=int(bs[end,s]);end=begin
            if s<0 and end>0:s=int(np.argmax(score[end]))
        return z

def fit_hsmm(X,n_states=3):
    t=time.perf_counter()
    try:
        m=ExplicitDurationGaussianHSMM(n_states=n_states).fit(X);s=m.predict(X)
        return RegimeResult('Explicit-duration HSMM',s,None,time.perf_counter()-t,m)
    except Exception as e:return RegimeResult('Explicit-duration HSMM',np.full(len(X),-1),None,time.perf_counter()-t,None,False,str(e))

## 3. Synthetic benchmark

In [ ]:
X_syn=StandardScaler().fit_transform(synthetic_df)
results=[fit_hmmlearn(X_syn),fit_pomegranate(X_syn),fit_hsmm(X_syn)]
for r in results:print(r.method,'available=',r.available,r.message)

In [ ]:
rows=[];aligned_sequences={}
for r in results:
    if not r.available:
        rows.append({'method':r.method,'available':False,'message':r.message});continue
    z,mapping=align_states(true_states,r.states);aligned_sequences[r.method]=z;L=run_lengths(z)
    rows.append({'method':r.method,'available':True,'accuracy':accuracy_score(true_states,z),'macro_f1':f1_score(true_states,z,average='macro'),'adjusted_rand':adjusted_rand_score(true_states,r.states),'switch_rate':switch_rate(z),'median_run_length':float(np.median(L)),'short_run_ratio_le_2':float(np.mean(L<=2)),'posterior_entropy':posterior_entropy(r.probabilities),'runtime_seconds':r.runtime_seconds,'mapping':mapping})
synthetic_scores=pd.DataFrame(rows);synthetic_scores

In [ ]:
fig,axes=plt.subplots(len(aligned_sequences)+1,1,figsize=(15,2.2*(len(aligned_sequences)+1)),sharex=True)
axes[0].step(synthetic_df.index,true_states,where='post');axes[0].set_ylabel('Truth')
for ax,(name,z) in zip(axes[1:],aligned_sequences.items()):ax.step(synthetic_df.index,z,where='post');ax.set_ylabel(name.split()[0])
plt.tight_layout();plt.show()

## 4. Load v12 Ricci rolling features

Set `RICCI_FEATURE_CSV` to a CSV produced by `rolling_feature_table(frames)`. If it is `None`, a proxy table is built so the notebook remains executable.

In [ ]:
RICCI_FEATURE_CSV=None
REAL_FEATURES=['avg_ricci','ricci_std','ricci_min','negative_edge_ratio','clusters','largest_component_ratio','density','component_entropy','edge_stability','total_edge_capital_flow','max_node_capital_share']
if RICCI_FEATURE_CSV:
    ricci_df=pd.read_csv(RICCI_FEATURE_CSV)
    if 'date' in ricci_df:ricci_df['date']=pd.to_datetime(ricci_df['date']);ricci_df=ricci_df.set_index('date')
else:
    ricci_df=synthetic_df.rename(columns={'edge_instability':'edge_stability'}).copy();ricci_df['ricci_std']=.12+.3*ricci_df.negative_edge_ratio.abs();ricci_df['ricci_min']=ricci_df.avg_ricci-.25;ricci_df['clusters']=np.clip(np.round(2+4*ricci_df.negative_edge_ratio),1,7);ricci_df['component_entropy']=np.clip(.3+ricci_df.negative_edge_ratio,0,None);ricci_df['total_edge_capital_flow']=np.exp(18+.8*ricci_df.density)
avail=[c for c in REAL_FEATURES if c in ricci_df]
ricci_features=ricci_df[avail].replace([np.inf,-np.inf],np.nan).interpolate(limit_direction='both').fillna(0)
ricci_features.head()

## 5. Walk-forward stability validation

In [ ]:
def walk_forward_benchmark(df,n_states=3,initial_train=120,test_block=30):
    X=StandardScaler().fit_transform(df);methods={'hmmlearn':fit_hmmlearn,'pomegranate':fit_pomegranate,'hsmm':fit_hsmm};M=[];S={}
    for name,fit in methods.items():
        pred=np.full(len(X),-1,int);times=[];fail=[]
        for train_end in range(initial_train,len(X),test_block):
            end=min(train_end+test_block,len(X));r=fit(X[:end],n_states);times.append(r.runtime_seconds)
            if r.available:pred[train_end:end]=r.states[train_end:end]
            else:fail.append(r.message)
        valid=pred>=0
        if not valid.any():M.append({'method':name,'available':False,'message':'; '.join(sorted(set(fail)))});continue
        z=pred[valid];L=run_lengths(z);S[name]=pred
        M.append({'method':name,'available':True,'evaluated_frames':int(valid.sum()),'switch_rate':switch_rate(z),'median_run_length':float(np.median(L)),'mean_run_length':float(np.mean(L)),'short_run_ratio_le_2':float(np.mean(L<=2)),'runtime_seconds_total':float(np.sum(times)),'failures':len(fail)})
    return pd.DataFrame(M),S
initial_train=min(120,max(30,len(ricci_features)//2))
walk_scores,walk_sequences=walk_forward_benchmark(ricci_features,initial_train=initial_train)
walk_scores

In [ ]:
if walk_sequences:
    fig,axes=plt.subplots(len(walk_sequences),1,figsize=(15,2.5*len(walk_sequences)),sharex=True)
    if len(walk_sequences)==1:axes=[axes]
    for ax,(name,z) in zip(axes,walk_sequences.items()):ax.step(ricci_features.index,z,where='post');ax.set_title(name);ax.set_yticks([-1,0,1,2])
    plt.tight_layout();plt.show()

## 6. Interpretation and selection

- Keep **hmmlearn** as the reproducible baseline.
- Select **pomegranate** only if its measured accuracy/stability or GPU execution provides a real advantage.
- Select **HSMM** when duration realism and suppression of one-frame flicker materially improve walk-forward results.
- Production v13 should keep at least two engines and display agreement/disagreement instead of declaring a single model universally correct.

In [ ]:
def summary_table(syn,walk):
    a=syn.copy();a['key']=a.method.str.lower().map(lambda x:'pomegranate' if 'pomegranate' in x else 'hsmm' if 'hsmm' in x else 'hmmlearn')
    return a.merge(walk,left_on='key',right_on='method',how='outer',suffixes=('_synthetic','_walk'))
summary_table(synthetic_scores,walk_scores)

## 7. Proposed v13 package layout

```text
ricci_finance/regimes/
├── base.py
├── hmmlearn_engine.py
├── pomegranate_engine.py
├── hsmm_engine.py
├── benchmark.py
└── consensus.py
```

The app should use a common `fit / predict / predict_proba` interface. HSMM posterior probability should remain unavailable until a proper forward-backward implementation is added; do not invent confidence values.

## References

- hmmlearn official API documentation for `GaussianHMM.fit`, `predict`, `predict_proba`, and `score`.
- pomegranate official HMM tutorial for 3-D sequence input, `DenseHMM`, fitting, and posterior probabilities.
- pomegranate project documentation for its PyTorch backend, GPU support, masked missing values, and dense/sparse HMM implementations.